# Step 1: Data Exploration & Preparation (EDA & Preprocessing)

**Chapter 2 Case Study** — Machine Learning in Healthcare (ET4248) — Project 3.4 "Rapid Heart Failure"

> This notebook is a **complete illustrative solution** applying Chapter 2
> theory (Section 2.3 EDA, 2.4 mathematical foundations, 2.5 Rubin's
> missing-data theory, 2.6 the 4-step leakage-free pipeline, 2.7 real-world
> clinical data conventions) to a real problem:
> [Heart Failure Clinical Records](https://archive.ics.uci.edu/dataset/519/heart+failure+clinical+records)
> (UCI #519, CC BY 4.0, 299 patients, 13 features).
>
> ⚠️ **If you were assigned Project 3.4 (graded coursework):** this
> notebook is a course illustration, NOT a valid submission for the
> assignment — read the Zero-Code Policy notice on the
> [Project 3.4 page](/en/du_an/suy_tim_risk_dxai) before using it.

In [1]:
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler

pd.set_option('display.width', 120)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def _load_heart_failure_data():
    """Load the local file (../data/...) if present (running inside a
    cloned HMYT repo); otherwise (opened standalone via Colab/Kaggle, no
    accompanying data/ folder) automatically download it from the public
    mirror on hmyt-book (Public repo, verified 2026-09-23)."""
    import os
    local_path = "../data/heart_failure_clinical_records_dataset.csv"
    remote_url = ("https://raw.githubusercontent.com/fossbk-spec/hmyt-book/gh-pages/"
                  "labs_chuyen_de/ch02_suy_tim_risk_dxai/data/"
                  "heart_failure_clinical_records_dataset.csv")
    path = local_path if os.path.exists(local_path) else remote_url
    if path == remote_url:
        print("[i] Local data not found — downloading from the public mirror: " + remote_url)
    return pd.read_csv(path).rename(columns={"death_event": "DEATH_EVENT"})


## 1. Load & Overview of Real Data (Section 2.3 — EDA)

Data is loaded directly from the UCI Machine Learning Repository (not
simulated — unlike the general-purpose Chapter 2 Code Lab, which uses
synthetic data).

In [2]:
df = _load_heart_failure_data()

print(f"Number of patients: {df.shape[0]}  |  Number of features (excluding label): {df.shape[1] - 1}")
df.head()


Number of patients: 299  |  Number of features (excluding label): 12


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       299 non-null    float64
 1   anaemia                   299 non-null    int64  
 2   creatinine_phosphokinase  299 non-null    int64  
 3   diabetes                  299 non-null    int64  
 4   ejection_fraction         299 non-null    int64  
 5   high_blood_pressure       299 non-null    int64  
 6   platelets                 299 non-null    float64
 7   serum_creatinine          299 non-null    float64
 8   serum_sodium              299 non-null    int64  
 9   sex                       299 non-null    int64  
 10  smoking                   299 non-null    int64  
 11  time                      299 non-null    int64  
 12  DEATH_EVENT               299 non-null    int64  
dtypes: float64(3), int64(10)
memory usage: 30.5 KB


**Observation:** all 13 columns are numeric (`int64`/`float64`) — there is
no string/categorical column requiring encoding, unlike the
general-purpose Chapter 2 Code Lab (which has a categorical `gender`
column requiring `OneHotEncoder`). This is a genuine difference between
the two problems, not an oversight.

## 2. Checking for Missing Data (Section 2.5 — Rubin's Theory)

In [4]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing.to_string())
print(f"\nTotal: {missing.sum()} missing values out of {df.size} data cells.")

Missing values per column:
age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0

Total: 0 missing values out of 3887 data cells.


**REAL result (not simulated):** the Heart Failure Clinical Records
dataset has **no missing values whatsoever** — this is a genuine property
of the dataset (already cleaned by Chicco & Jurman before publication),
unlike the raw EHR data used in the general-purpose Chapter 2 Code Lab.

Since Rubin's theory (MCAR/MAR/MNAR — Section 2.5) has nothing to
illustrate directly on the original dataset, the section below creates
**a copy with artificially injected missingness** (clearly an additional
illustration, NOT a real property of the original data) to practice the
two missingness mechanisms most common in clinical settings, following
the 4-step leakage-free pipeline (Section 2.6).

In [5]:
df_demo = df.copy()

# MCAR illustration (Missing Completely At Random): a random device error
# drops 8% of serum_sodium values, independent of any other variable.
rng = np.random.default_rng(RANDOM_STATE)
mcar_mask = rng.random(len(df_demo)) < 0.08
df_demo.loc[mcar_mask, 'serum_sodium'] = np.nan

# MAR illustration (Missing At Random): physicians order a
# creatinine_phosphokinase (cardiac enzyme) test depending on the observed
# age and ejection_fraction — younger patients with normal ejection_fraction
# are ordered the test less often.
mar_prob = 1 / (1 + np.exp(-(0.04 * (df_demo['age'] - 60) - 0.05 * (df_demo['ejection_fraction'] - 38))))
mar_mask = rng.random(len(df_demo)) > mar_prob
df_demo.loc[mar_mask, 'creatinine_phosphokinase'] = np.nan

print("Artificial missingness rate (illustration only, not the original data):")
print((df_demo[['serum_sodium', 'creatinine_phosphokinase']].isnull().mean() * 100).round(2).astype(str) + ' %')

Artificial missingness rate (illustration only, not the original data):
serum_sodium                7.02 %
creatinine_phosphokinase    50.5 %
dtype: str


## 3. Label Imbalance Analysis (Section 2.3 — EDA)

In [6]:
counts = df['DEATH_EVENT'].value_counts().sort_index()
ratios = df['DEATH_EVENT'].value_counts(normalize=True).sort_index()
print("DEATH_EVENT label distribution (0 = Survived, 1 = Death):")
for label, cnt, ratio in zip(counts.index, counts.values, ratios.values):
    name = 'Survived' if label == 0 else 'Death'
    print(f"  {label} ({name}): {cnt} patients ({ratio*100:.1f}%)")

DEATH_EVENT label distribution (0 = Survived, 1 = Death):
  0 (Survived): 203 patients (67.9%)
  1 (Death): 96 patients (32.1%)


**REAL result:** 203 survived (67.9%) / 96 died (32.1%) — a moderate
imbalance (ratio ~2.1:1), matching the description in the original paper
`chicco2020machine`. This is why Step 3 applies SMOTE.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
counts.rename({0: 'Survived', 1: 'Death'}).plot(kind='bar', ax=axes[0], color=['#4C72B0', '#C44E52'])
axes[0].set_title('DEATH_EVENT label distribution (N=299)')
axes[0].set_ylabel('Number of patients')
axes[0].tick_params(axis='x', rotation=0)

corr = df.corr(numeric_only=True)
sns.heatmap(corr, ax=axes[1], cmap='coolwarm', center=0, annot=False, cbar_kws={'shrink': 0.8})
axes[1].set_title('Correlation matrix — 13 features')

plt.tight_layout()
plt.savefig('../figures/01_eda_overview_en.png', dpi=110, bbox_inches='tight')
plt.show()
print("[+] Saved ../figures/01_eda_overview_en.png")

[+] Saved ../figures/01_eda_overview_en.png


C:\Users\hoang\AppData\Local\Temp\ipykernel_48952\2302488453.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
print("Top 5 features by absolute correlation with DEATH_EVENT:")
print(corr['DEATH_EVENT'].drop('DEATH_EVENT').abs().sort_values(ascending=False).head(5).round(3).to_string())

Top 5 features by absolute correlation with DEATH_EVENT:
time                 0.527
serum_creatinine     0.294
ejection_fraction    0.269
age                  0.254
serum_sodium         0.195


## 4. Leakage-Free Train/Test Split (Section 2.6 — 4-Step Pipeline, Step 1)

*Reflection question (from the original assignment):* Why should you only
call `fit` on the Train set and never on the Test set?
**Answer:** if you `fit` (compute the mean/std for `StandardScaler`, or fit
any preprocessing step) on the whole dataset (including Test), statistical
information from the Test set "leaks" into the Train set's normalization —
the model indirectly "sees" the distribution it will later be evaluated
on, which makes the measured performance artificially optimistic (it no
longer reflects genuine generalization to entirely new patients).

In [9]:
FEATURE_COLS = [c for c in df.columns if c != 'DEATH_EVENT']
X = df[FEATURE_COLS]
y = df['DEATH_EVENT'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]} patients | Test: {X_test.shape[0]} patients")
print(f"Train death rate: {y_train.mean()*100:.1f}%  |  Test death rate: {y_test.mean()*100:.1f}%")
print("(The two rates are close thanks to stratify=y — the original label distribution is preserved in both sets)")

Train: 239 patients | Test: 60 patients
Train death rate: 32.2%  |  Test death rate: 31.7%
(The two rates are close thanks to stratify=y — the original label distribution is preserved in both sets)


## 5. Feature Scaling — `fit` on Train Only (Section 2.4 — Mathematical Foundations)

In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on Train
X_test_scaled = scaler.transform(X_test)          # transform ONLY on Test (using the mean/std learned from Train)

print("Mean/Std learned from Train (first 5 features):")
for name, m, s in list(zip(FEATURE_COLS, scaler.mean_, scaler.scale_))[:5]:
    print(f"  {name:28s}  mean={m:10.3f}  std={s:10.3f}")

Mean/Std learned from Train (first 5 features):
  age                           mean=    61.073  std=    11.420
  anaemia                       mean=     0.448  std=     0.497
  creatinine_phosphokinase      mean=   602.791  std=  1010.243
  diabetes                      mean=     0.448  std=     0.497
  ejection_fraction             mean=    37.887  std=    11.970


## 6. Full Leakage-Free Pipeline (Section 2.6 — illustrated on the demo dataset with missing values)

Applying the same `ColumnTransformer` + `Pipeline` framework as the
general-purpose Chapter 2 Code Lab, but this time on `df_demo` (the
version with artificial missingness from Section 2) to practice both
imputation and scaling within a single `fit` call on Train.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

X_demo = df_demo[FEATURE_COLS]
y_demo = df_demo['DEATH_EVENT'].values
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_demo, y_demo, test_size=0.20, stratify=y_demo, random_state=RANDOM_STATE
)

# The 2 columns with artificial missingness need their own Imputer; the rest are scaled directly
missing_cols = ['serum_sodium', 'creatinine_phosphokinase']
clean_cols = [c for c in FEATURE_COLS if c not in missing_cols]

missing_pipeline = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler()),
])
clean_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
])
preprocessor = ColumnTransformer(transformers=[
    ('missing', missing_pipeline, missing_cols),
    ('clean', clean_pipeline, clean_cols),
])
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
])

full_pipeline.fit(Xd_train, yd_train)   # a SINGLE fit call on Train — Imputer + Scaler + Model
yd_prob = full_pipeline.predict_proba(Xd_test)[:, 1]
yd_pred = full_pipeline.predict(Xd_test)

print(f"ROC-AUC (on the demo dataset with missing values, after KNNImputer): {roc_auc_score(yd_test, yd_prob):.4f}")
print(classification_report(yd_test, yd_pred, target_names=['Survived', 'Death']))

ROC-AUC (on the demo dataset with missing values, after KNNImputer): 0.8588
              precision    recall  f1-score   support

    Survived       0.84      0.93      0.88        41
       Death       0.80      0.63      0.71        19

    accuracy                           0.83        60
   macro avg       0.82      0.78      0.79        60
weighted avg       0.83      0.83      0.83        60



## 7. Step 1 Summary

- The real data has **no missing values**, consistent with the original
  paper `chicco2020machine` using all 13 features without imputation.
- Real label imbalance is **32.1% deaths** — addressed in Step 3 (SMOTE).
- `X_train`/`X_test`/`y_train`/`y_test` are built the standard way
  (leakage-free, `stratify=True`, `random_state=42`) — Step 2 reuses this
  exact split.
- An additional complete leakage-free pipeline (Imputer + Scaler + Model
  in a single `fit`) was demonstrated on data with artificial missingness,
  following the Rubin's-theory framework from Section 2.5.

**Next:** [`2_baseline.ipynb`](./2_baseline.ipynb) — Step 2, reproducing
the `chicco2020machine` result (Accuracy 74.0%, MCC 0.384).